In [ ]:
import pandas as pd
import numpy as np
from statics import *
import seaborn as sns
import matplotlib.pyplot as plt
# https://data.crunchbase.com/docs/data-dictionary

In [ ]:
pd.set_option('display.max_columns', None)  # Show all columns when displaying DataFrames

In [ ]:
# load raw data
org_df_raw = pd.read_csv('20251012_bulk_export/organizations.csv')
org_des_df_raw = pd.read_csv('20251012_bulk_export/organization_descriptions.csv')
fund_rounds_df_raw = pd.read_csv('20251012_bulk_export/funding_rounds.csv')
llm_cls_df_raw = pd.read_csv('crunchbase_companies_final.csv')

# add LLM classification 

# narrow down the scope
- North America (CAN + USA)
- Company roles: please keep only “company” and “company,investor”  (I.e., exclude “school”, “investor”, “school,company”, etc.)
- Founding date (founded_on) 2005 or later (if missing, you can keep the observation for now)
- If company “status” is closed: closing date (closed_on) 2019 or late

In [ ]:
llm_cls_df = llm_cls_df_raw[['uuid','llm_classification','llm_confidence']]
def classify_platform(row):
    if row['llm_classification'] == 'platform' and row['llm_confidence'] >= 95:
        return 'Narrow'
    elif row['llm_classification'] == 'platform' and row['llm_confidence'] >= 85:
        return 'Broad'
    elif row['llm_classification'] == 'non-platform' and row['llm_confidence'] >= 60:
        return 'Non-Platform'
llm_cls_df['is_platform'] = llm_cls_df.apply(classify_platform, axis=1)

In [ ]:
org_df = org_df_raw[org_df_raw['country_code'].isin(['USA', 'CAN'])]  # filter to US and Canada only
org_df = org_df[org_df['roles'].isin(['company', 'company,investor'])]  # filter to keep only organizations that are not investors
# fileter on founded_on >=2005-01-01
org_df['founded_on'] = pd.to_datetime(org_df['founded_on'], errors='coerce')
org_df = org_df[org_df['founded_on'] >= '2005-01-01']
# exclude companies with  status 'closed' and close_on before 2019-01-01
org_df['close_on'] = pd.to_datetime(org_df['closed_on'], errors='coerce')
org_df = org_df[~((org_df['status'] == 'closed') & (org_df['close_on'] < '2019-01-01'))]

# add description info 
org_df = pd.merge(org_des_df_raw[['uuid','description']], org_df, on='uuid',how='right')
# add llm classification info
org_df = pd.merge(org_df, llm_cls_df[['uuid','is_platform']], on='uuid', how='left')


In [ ]:
org_df.head()

In [ ]:
cats = org_df['category_list']
cats_exploded = cats.str.split(',').explode().str.strip()
all_categories = cats_exploded.dropna().unique()
len(all_categories)

# Add fund rouds info
- Is vc funds
- Startups that are private but have not raised a funding round (or recorded any other event) in 5 years can be coded as “inactive”
- Add age of companies

In [ ]:
print(f"Number of organizations after filtering: {org_df['uuid'].nunique()}")
# Add fund rouds info
print(f"Number of funding rounds before filtering: {fund_rounds_df_raw['org_uuid'].nunique()}")


In [ ]:
fund_rounds_df = fund_rounds_df_raw[['uuid', 'org_uuid', 'name', 'investment_type', 'announced_on', 'raised_amount_usd']].copy()
fund_rounds_df.columns = [f"FR_{c}" for c in fund_rounds_df.columns]

# join org_df with fund_rounds_df_raw on 'uuid' and 'organization_uuid', # TODO companies without funding??????
df_org_fund = pd.merge(org_df, fund_rounds_df, left_on='uuid', right_on='FR_org_uuid', how='inner')

# exclude all organizations where the first funding round of any type was before 2005
df_org_fund['FR_announced_on'] = pd.to_datetime(df_org_fund['FR_announced_on'], errors='coerce')

df_org_fund = df_org_fund.sort_values('FR_announced_on')
first_funding = df_org_fund.groupby('uuid')['FR_announced_on'].min().reset_index()
drop_orgs = first_funding[first_funding['FR_announced_on'] < '2005-01-01']['uuid'].unique()
# filter org_df (keep orgs with no funding record or first funding >= 2005)
df_org_fund = df_org_fund[~df_org_fund['uuid'].isin(drop_orgs)]

# for private companies, if last funding round is more than 5 years ago, code as inactive
df_org_fund['FR_announced_on'] = pd.to_datetime(df_org_fund['FR_announced_on'], errors='coerce')
last_funding = df_org_fund[df_org_fund['status']=='operating'].groupby('uuid')['FR_announced_on'].max().reset_index()
last_funding['is_inactive'] = (last_funding['FR_announced_on'] < pd.Timestamp.now() - pd.DateOffset(years=5)).astype(int)
df_org_fund = pd.merge(df_org_fund, last_funding[['uuid', 'is_inactive']], on='uuid', how='left')


df_org_fund['is_vc_funded'] = df_org_fund['FR_investment_type'].isin(VC_FUND_TYPES).astype(int)

# add age of companies based on founded_on date and current date
df_org_fund['founded_on'] = pd.to_datetime(df_org_fund['founded_on'], errors='coerce')
df_org_fund['company_age'] = ((pd.Timestamp.now() - df_org_fund['founded_on']).dt.days / 365).astype(int)

In [ ]:
org_df_raw.columns

In [ ]:
df_org_fund.head()

In [ ]:
df_org_fund['is_inactive'].value_counts()

In [ ]:
df_org_fund['is_vc_funded'].value_counts()

In [ ]:
df_org_fund.head()

# (not available anymore)!!!  

# Apply key words identification of platform and non platform 
- baded on category list 
- based on description

### Distinguish non-platform businesses between:   
-“Digital” companies (software)   
-“Non-digital” companies: robotics, industrials, hardware, consumer staples, biotech, pharmaceuticals – anything physical   
-(So, there will be two “control groups”. Don’t worry if this distinction is very imperfect – even something approximate will be fine.)

In [ ]:
df_org_fund['num_category_platform'] = df_org_fund['category_list'].apply(
    lambda x: sum(1 for cat in platform_categories if cat in x.split(',')) if pd.notna(x) else 0)


In [ ]:
df_org_fund['num_category_nonplatform'] = df_org_fund['category_list'].apply(
    lambda x: sum(1 for cat in non_platform_categories if cat in x.split(',')) if pd.notna(x) else 0)


In [ ]:
df_org_fund['num_keyword_platform'] = df_org_fund['description'].apply(
    lambda x: sum(1 for kw in platform_keywords if kw in x.lower()) if pd.notna(x) else 0
)
df_org_fund['num_keyword_nonplatform'] = df_org_fund['description'].apply(
    lambda x: sum(1 for kw in non_platform_keywords if kw in x.lower()) if pd.notna(x) else 0)

In [ ]:
# df_org_fund['num_category_digital_nonplatform'] = df_org_fund['category_list'].apply(
#     lambda x: sum(1 for cat in digital_nonplatform_categories if cat in x.split(',')) if pd.notna(x) else 0)

# df_org_fund['num_category_nonditigal_nonplatform'] = df_org_fund['category_list'].apply(
#     lambda x: sum(1 for cat in nondigital_nonplatform_categories if cat in x.split(',')) if pd.notna(x) else 0)

df_org_fund['num_keyword_digital_nonplatform'] = df_org_fund['category_list'].apply(
    lambda x: sum(1 for kw in digital_nonplatform_keywords if kw in x.lower()) if pd.notna(x) else 0
)

df_org_fund['num_keyword_nonditigal_nonplatform'] = df_org_fund['category_list'].apply(
  lambda x: sum(1 for kw in nondigital_nonplatform_keywords if kw in x.lower()) if pd.notna(x) else 0
)



In [ ]:
def check_platform_strict(row):
    if row['num_category_platform'] > row['num_category_nonplatform'] and row['num_keyword_platform'] > row['num_keyword_nonplatform']:
        return 1
    else:
        return 0

In [ ]:
def check_platform_weak(row):
    if row['num_category_platform'] > row['num_category_nonplatform'] or row['num_keyword_platform'] > row['num_keyword_nonplatform']:
        return 1
    else:
        return 0

In [ ]:
def check_platform_weak(row):
    if row['num_category_platform'] > row['num_category_nonplatform'] or row['num_keyword_platform'] > row['num_keyword_nonplatform']:
        return 1
    else:
        return 0

In [ ]:
def check_digital_nonplatform(row):
    if row['is_platform']=='Non-Platform':
        if row['num_keyword_digital_nonplatform'] >= row['num_keyword_nonditigal_nonplatform']:
            return 1
        else:
            return 0
    else:
        return -1 

In [ ]:
df_org_fund['is_platform_strict'] = df_org_fund.apply(lambda row: check_platform_strict(row), axis=1)
df_org_fund['is_platform_weak'] = df_org_fund.apply(lambda row: check_platform_weak(row), axis=1)

In [ ]:
df_org_fund['is_digital_nonplatform'] = df_org_fund.apply(lambda row: check_digital_nonplatform(row), axis=1)

In [ ]:
df_org_fund[['is_platform','is_digital_nonplatform']].value_counts()

In [ ]:
df_org_fund.head()

In [ ]:
df_org_fund.columns

In [ ]:
df_org_grouped = df_org_fund.groupby('uuid', as_index=False).first()
df_org_grouped['first_round_on'] = df_org_grouped['FR_announced_on']
df_org_grouped.drop(columns=['FR_uuid', 'FR_org_uuid', 'FR_name', 'FR_investment_type', 'FR_announced_on', 'FR_raised_amount_usd'], inplace=True)

In [ ]:
df_org_grouped['description']

In [ ]:
pd.set_option('display.max_colwidth', None)
df_org_grouped[['name', 'description', 'is_platform', 'is_digital_nonplatform', 'num_keyword_digital_nonplatform', 'num_keyword_nonditigal_nonplatform']].head(20)

In [ ]:
df_simple = df_org_fund[['uuid', 'description','name',
                        'is_platform', 'is_digital_nonplatform', 'company_age']].copy()

In [ ]:
df_company = df_simple.groupby('uuid', as_index=False).first()

In [ ]:
df_company['is_digital_nonplatform'].value_counts()

In [ ]:
df_company.head()

In [ ]:
# df_company.to_csv('crunchbase_orgs_platform_classification.csv', index=False)

In [ ]:
platform_narrow = df_org_grouped[df_org_grouped['is_platform']=='Narrow']
platform_broad = df_org_grouped[df_org_grouped['is_platform'] == 'Broad']
nonplatform = df_org_grouped[df_org_grouped['is_platform'] == 'Non-Platform']

In [ ]:
df_org_fund['is_platform'].value_counts()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 8))

# --- (1,1) strict - num_funding_rounds ---
sns.boxplot(
    data=df_org_grouped,
    x='is_platform',
    y='num_funding_rounds',
    showfliers=False,
    ax=axes[0]
)
# axes[0].set_xticklabels(['Non-platform', 'Platform'])
axes[0].set_title('Funding Rounds by Firm Type ')

# --- (1,2) strict - total_funding_usd ---
sns.boxplot(
    data=df_org_grouped,
    x='is_platform',
    y='total_funding_usd',
    showfliers=False,
    ax=axes[1]
)
axes[1].set_yscale('log')
# axes[1].set_xticklabels(['Non-platform', 'Platform'])
axes[1].set_title('Total Funding (log scale) by Firm Type')


# --- Layout ---
plt.tight_layout()
plt.suptitle('Funding Comparison Between Platform and Non-platform Firms', fontsize=14, y=1.02)
plt.show()

In [ ]:


fig, axes = plt.subplots(1, 1, figsize=(12, 5), sharey=True)

# --- strict ---
sns.kdeplot(np.log1p(platform_narrow['total_funding_usd']),
            label='Platform (Narrow)', fill=True, alpha=0.5, ax=axes)
sns.kdeplot(np.log1p(platform_broad['total_funding_usd']),
            label='Platform', fill=True, alpha=0.5, ax=axes)
sns.kdeplot(np.log1p(nonplatform['total_funding_usd']),
            label='Non-platform', fill=True, alpha=0.5, ax=axes)
axes.set_title('strict Logic')
axes.set_xlabel('log(1 + Total Funding USD)')
axes.set_ylabel('Density')
axes.legend()


plt.suptitle('Distribution of Total Funding (log1p scale)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Ensure integer rounds
df_org_grouped['num_funding_rounds'] = df_org_grouped['num_funding_rounds'].astype(int)

fig, axes = plt.subplots(1, 1, figsize=(14, 6), sharey=True)

# --- (1) strict ---
sns.stripplot(
    data=df_org_grouped,
    x='num_funding_rounds',
    y='total_funding_usd',
    hue='is_platform',
    dodge=True,
    jitter=0.25,
    alpha=0.5,
    size=2,
    ax=axes
)
axes.set_yscale('log')
axes.set_xticks(range(0, 36, 2))
axes.set_xlabel('Number of Funding Rounds')
axes.set_ylabel('Total Funding (USD, log scale)')
axes.set_title('Funding vs Rounds (strict)')
axes.legend(title='Platform')


plt.tight_layout()
plt.suptitle('Funding vs Rounds by Platform Type', fontsize=14, y=1.02)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(12, 5), sharey=True)

# strict
avg_funding_strict = (
    df_org_grouped
    .groupby(['num_funding_rounds', 'is_platform'])['total_funding_usd']
    .median()
    .reset_index()
)

sns.lineplot(
    data=avg_funding_strict,
    x='num_funding_rounds',
    y='total_funding_usd',
    hue='is_platform',
    marker='o',
    ax=axes
)
axes.set_yscale('log')
axes.set_title('strict Logic')
axes.set_xlabel('Number of Funding Rounds')
axes.set_ylabel('Median Total Funding (USD, log scale)')
axes.legend(title='Platform')


plt.suptitle('Median Total Funding vs Number of Rounds ', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

# Descriptives
- build a panel of firms at the quarterly level that keeps track of the number of funding rounds the firm has received, the amount of funding, exits (“status” of firm) etc. 
- distinguish non-platform businesses between:   
    -“Digital” companies (software)   
    -“Non-digital” companies: robotics, industrials, hardware, consumer staples, biotech, pharmaceuticals – anything physical   
    -(So, there will be two “control groups”. Don’t worry if this distinction is very imperfect – even something approximate will be fine.)

- Startups that are private but have not raised a funding round (or recorded any other event) in 5 years can be coded as “inactive”

### 1. build a panel of firms at the quarterly level that keeps track of the number of funding rounds the firm has received, the amount of funding, exits (“status” of firm) etc. 

In [ ]:
df_quarterly = df_org_fund[['uuid', 'name', 'type','created_at', 'num_funding_rounds', 'total_funding_usd', 'founded_on', 'last_funding_on','num_exits', 'FR_name', 'FR_investment_type', 'FR_announced_on', 'FR_raised_amount_usd', 'is_vc_funded', 'is_platform']].copy()
# Convert relevant dates to datetime if not already
df_quarterly['FR_announced_on'] = pd.to_datetime(df_quarterly['FR_announced_on'])

# Create quarter column using FR_announced_on
df_quarterly['quarter'] = df_quarterly['FR_announced_on'].dt.to_period('Q')

In [ ]:
# Group by uuid and quarter to calculate funding metrics
quarterly_funding = df_quarterly.groupby(['uuid', 'quarter']).agg({
    'name': 'first',
    'type': 'first',
    'created_at': 'first',
    'num_funding_rounds': 'count',
    'total_funding_usd': 'first',
    'founded_on': 'first',
    'last_funding_on': 'first',
    'FR_name': lambda x: list(x),  # collect names as list
    'FR_investment_type': lambda x: list(x),  # collect names as list
    'FR_raised_amount_usd': 'sum',
    'num_exits': lambda x: x.fillna(0).sum(),  # sum exits, treating NaN as 0
    'FR_announced_on': lambda x: list(x),  # collect announced dates as list
    'is_vc_funded': 'first',
    'is_platform': 'first',

}).reset_index()

# Sort by uuid and quarter
quarterly_funding['FR_num_round'] = quarterly_funding['FR_name'].apply(len)
quarterly_funding = quarterly_funding.sort_values(['uuid', 'quarter'])

# Add cumulative metrics
quarterly_funding['cum_num_rounds'] = quarterly_funding.groupby('uuid')['FR_num_round'].cumsum()
quarterly_funding['cum_funding'] = quarterly_funding.groupby('uuid')['FR_raised_amount_usd'].cumsum()
quarterly_funding['cum_exits'] = quarterly_funding.groupby('uuid')['num_exits'].cumsum()

In [ ]:
quarterly_funding[quarterly_funding['uuid']=='845a7147-8b85-093a-54f4-d0f7e891aa20']

In [ ]:
quarterly_funding['quarter_dt'] = pd.PeriodIndex(quarterly_funding['quarter'], freq='Q').to_timestamp()
exit_trend = quarterly_funding.groupby(['quarter_dt','is_platform'])['num_exits'].sum().reset_index()
sns.lineplot(data=exit_trend, x='quarter_dt', y='num_exits', hue='is_platform')
plt.title('Quarterly Exits (M&A/IPO) by Firm Type')

In [ ]:

fig, axes = plt.subplots(1, 1, figsize=(14, 6), sharey=True)

# ---------------- strict ----------------
sns.lineplot(
    data=quarterly_funding, 
    x='quarter_dt', 
    y='cum_funding', 
    hue='is_platform',
    estimator='median',
    ci=None,
    ax=axes,
    marker='o'
)
axes.set_title('Median Cumulative Funding over Time')
axes.set_xlabel('Quarter')
axes.set_ylabel('Cumulative Funding (USD, log scale)')
axes.set_yscale('log')
axes.legend(title='Platform')
axes.grid(alpha=0.3)
# Adjust layout
plt.suptitle('Cumulative Funding over Time', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

In [ ]:

quarterly_funding['quarter_dt'] = pd.PeriodIndex(quarterly_funding['quarter'], freq='Q').to_timestamp()
# Create subplots
fig, axes = plt.subplots(1, 1, figsize=(14, 6), sharey=True)

# ---------------- strict ----------------
sns.lineplot(
    data=quarterly_funding, 
    x='quarter_dt', 
    y='cum_num_rounds', 
    hue='is_platform',
    estimator='median',
    ci=None,
    ax=axes,
    marker='o'
)
axes.set_title('Median Cumulative rounds over Time')
axes.set_xlabel('Quarter')
axes.set_ylabel('Cumulative (rounds')
axes.legend(title='Platform')
axes.grid(alpha=0.3)
# Adjust layout
plt.suptitle('Cumulative round over Time', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

Conclusion: 


## Interesting descriptives:
### Age
1. Conditional on startup age (for example, for all startups of age 5 years), what is the average number of funding rounds for platform vs. non-platform businesses? (Start with “VC only”; later could separately study all funding rounds) Total amount of funding for platform vs. non-platform businesses?
2. “Round cadence”: in a given company age, (number of rounds since start) / (years since first funding round)  Separately for platform vs. control groups
3. Conditional on startup age (for example, all startups of age 5 years), what is the rate of 
   - Being closed / inactive,
   - Being acquired,
   - Having gone public?

In [ ]:
# for item 3 above 
df_company_age_status= df_org_grouped[['uuid', 'name', 'company_age','status', 'is_inactive', 'is_vc_funded', 'is_platform', 'is_digital_nonplatform']].copy()
df_company_age_status['new_status'] = np.where(
    (df_company_age_status['is_inactive']==1 | df_company_age_status['status'].isin(['closed'])),
    'close/inactive',
    df_company_age_status['status']
)

In [ ]:
sns.displot(
    data=df_company_age_status,
    x='company_age',
    hue='new_status',
    col='is_platform',
    kind='hist',
    stat='proportion',
    multiple='fill',
    height=5,
    aspect=1.3
)

In [ ]:
status_counts = (
    df_company_age_status
    .groupby(['company_age', 'is_platform', 'new_status'])
    .size()
    .reset_index(name='count')
)
status_counts['percent'] = (
    status_counts.groupby(['company_age', 'is_platform'])['count']
    .transform(lambda x: x / x.sum())
)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = status_counts.copy()

pivot = df.pivot_table(
    index=["company_age", "is_platform"],
    columns="new_status",
    values="percent",
    fill_value=0,
).reset_index()

statuses = pivot.columns[2:] 

ages = sorted(pivot["company_age"].unique())
platforms = ['Narrow', 'Broad', 'Non-Platform']
bar_width = 0.4


cmap = plt.cm.get_cmap("tab10")
colors = {status: cmap(i) for i, status in enumerate(statuses)}

fig, ax = plt.subplots(figsize=(14, 6))
x_base = np.arange(len(ages))

for p_idx, p in enumerate(platforms):

    subset = pivot[pivot["is_platform"] == p]
    x_pos = x_base + (p_idx - 0.5) * bar_width
    bottom = np.zeros(len(subset))

    for st in statuses:
        vals = subset[st].values
        ax.bar(
            x_pos, vals, bar_width,
            bottom=bottom,
            color=colors[st],         
            label=st if p_idx == 0 else None   
        )
        bottom += vals

ax.set_xticks(x_base)
ax.set_xticklabels(ages)
ax.set_xlabel("Company Age")
ax.set_ylabel("Percent")
ax.set_title("Status Distribution by Age (Platform vs Non-platform)")
ax.legend(title="Status")

plt.tight_layout()
plt.show()

In [ ]:
df_company_age = df_org_grouped[['uuid', 'name','short_description', 'created_at', 'company_age','num_funding_rounds', 'first_round_on','total_funding_usd', 'num_exits', 'is_inactive', 'is_vc_funded', 'is_platform', 'is_digital_nonplatform']].copy()
# only consider vc funded companies
df_company_age = df_company_age[df_company_age['is_vc_funded'] == 1]
today = pd.Timestamp.today()
df_company_age['years_since_first_round'] = ((today - df_company_age['first_round_on']).dt.days / 365)
df_company_age['round_cadence'] = df_company_age['num_funding_rounds']/df_company_age['years_since_first_round']

In [ ]:
plt.figure(figsize=(15, 8))
sns.boxplot(
    data=df_company_age,
    hue='is_platform',
    y='round_cadence',
    x='company_age'
)
plt.yscale('log')
plt.title("Funding Round Cadence Distribution by Investment Type (Platform vs Non-platform)")
plt.xlabel("Investment Type")
plt.ylabel("Round Cadence (years, log scale)")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.kdeplot(
    data=df_company_age,
    x='round_cadence',
    hue='is_platform',
    log_scale=True,
    fill=True,
    common_norm=False
)
plt.title("Distribution of Funding Round Cadence")
plt.xlabel("Round Cadence (log scale)")
plt.show()

In [ ]:
platform_col = 'is_platform'
df_age_grouped = df_company_age.groupby(['company_age', platform_col]).agg({
    'uuid': 'count',
    'num_funding_rounds': 'mean',
    'total_funding_usd': 'mean',
    'num_exits': 'mean',
}).reset_index()
df_age_grouped = df_age_grouped.fillna(0)

In [ ]:
# plot lines for num_funding_rounds, total_funding_usd, num_exits vs company_age
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.lineplot(
    data=df_age_grouped,
    x='company_age',
    y='num_funding_rounds',
    hue=platform_col,
    marker='o',
    ax=axes[0]
)
axes[0].set_title('Average Number of Funding Rounds vs Company Age')
axes[0].set_xlabel('Company Age (years)')
axes[0].set_ylabel('Average Number of Funding Rounds')
# axes[0].legend(title='Platform', labels=['Non-platform','Platform'])
sns.lineplot(
    data=df_age_grouped,
    x='company_age',
    y='total_funding_usd',
    hue=platform_col,
    marker='o',
    ax=axes[1]
)
axes[1].set_yscale('log')
axes[1].set_title('Average Total Funding (log scale) vs Company Age')
axes[1].set_xlabel('Company Age (years)')
axes[1].set_ylabel('Average Total Funding (USD, log scale)')
# axes[1].legend(title='Platform', labels=['Non-platform','Platform'])
sns.lineplot(
    data=df_age_grouped,
    x='company_age',
    y='num_exits',
    hue=platform_col,
    marker='o',
    ax=axes[2]
)
axes[2].set_title('Average Number of Exits vs Company Age')
axes[2].set_xlabel('Company Age (years)')
axes[2].set_ylabel('Average Number of Exits')
# axes[2].legend(title='Platform', labels=['Non-platform','Platform'])
plt.suptitle('Company Age vs Funding and Exits by Platform Type', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

- What is the average and median size of funding rounds of types series_a, series_b, series_c for platform vs. non-platform businesses?

In [ ]:
df_fund_rounds = df_org_fund[['FR_org_uuid', 'name', 'FR_investment_type', 'FR_raised_amount_usd', 'is_platform']].copy()
# keep only some founds 
keep_investment_types = ['angel', 'pre_seed', 'seed', 'series_a', 'series_b', 'series_c','series_d','series_e','series_f','series_g','series_h','series_i', 'series_j']
df_fund_rounds = df_fund_rounds[df_fund_rounds['FR_investment_type'].isin(keep_investment_types)]
df_fund_rounds['FR_investment_type'] = pd.Categorical(
    df_fund_rounds['FR_investment_type'],
    categories=keep_investment_types,
    ordered=True
)

In [ ]:
plt.figure(figsize=(15, 8))
sns.boxplot(
    data=df_fund_rounds[df_fund_rounds['FR_investment_type'].isin(keep_investment_types)],
    x='FR_investment_type',
    y='FR_raised_amount_usd',
    hue='is_platform'
)
plt.yscale('log')
plt.title("Funding Amount Distribution by Investment Type (Platform vs Non-platform)")
plt.xlabel("Investment Type")
plt.ylabel("Funding Amount (USD, log scale)")
plt.xticks(rotation=45)
plt.show()

In [ ]:
platform_col = 'is_platform'
df_rounds_grouped = df_fund_rounds.groupby(['FR_investment_type', platform_col]).agg({
    'FR_org_uuid': 'count',
    'FR_raised_amount_usd': ['mean',lambda x: x.quantile(0.25), 'median',lambda x: x.quantile(0.75)],
}).reset_index()
df_rounds_grouped.columns = ['FR_investment_type', platform_col, 'company_count', 'raised_amount_mean', 'raised_amount_25', 'raised_amount_median', 'raised_amount_75']

In [ ]:
df_rounds_grouped

In [ ]:
quant_long = df_rounds_grouped.melt(
    id_vars=['FR_investment_type', 'is_platform'],
    value_vars=['raised_amount_mean', 'raised_amount_25', 'raised_amount_median', 'raised_amount_75'],
    var_name='stat',
    value_name='amount'
)
plt.figure(figsize=(10,6))
sns.barplot(
    data=quant_long,
    x='FR_investment_type',
    y='amount',
    hue='stat',
)
plt.yscale('log')
plt.xticks(rotation=45)
plt.title("Mean / Median / P25 / P75")
plt.ylabel("Amount (USD, log scale)")
plt.show()

Conditional on going public, what is the
- Amount of funding,
- Number of rounds,
- Years since first round for platform vs. non-platform businesses?

In [ ]:
df_ipo = df_org_grouped[['uuid', 'name','short_description', 'created_at', 'status','company_age','num_funding_rounds', 'first_round_on','total_funding_usd', 'num_exits', 'is_inactive', 'is_vc_funded', 'is_platform', 'is_digital_nonplatform']].copy()
df_ipo = df_ipo[df_ipo['status']=='ipo']
df_ipo['years_since_first_round'] = ((today - df_ipo['first_round_on']).dt.days / 365)
platform_col = 'is_platform'

In [ ]:
df_ipo.head()

In [ ]:
df_ipo[platform_col].value_counts()

In [ ]:

fig, axs = plt.subplots(1, 2, figsize=(14, 6))

# --- Boxplot ---
sns.boxplot(
    data=df_ipo,
    y='num_funding_rounds',
    x=platform_col,
    ax=axs[0]
)
axs[0].set_title("Funding rounds Distribution by Investment Type (Platform vs Non-platform)")
axs[0].set_xlabel("Investment Type")
axs[0].set_ylabel("number of rounds")

# --- KDE plot ---
sns.kdeplot(
    data=df_ipo,
    x='num_funding_rounds',
    hue=platform_col,
    fill=True,
    common_norm=False,
    ax=axs[1]
)
axs[1].set_title("Distribution of number of Funding Round of public company")
axs[1].set_xlabel("number of funding rounds")
axs[1].set_ylabel("Density")

plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# ---------------------------------------------------
# 1) BOX PLOT — Total Funding
# ---------------------------------------------------
sns.boxplot(
    data=df_ipo,
    y='total_funding_usd',
    x=platform_col,
    ax=axs[0]
)
axs[0].set_title("Total Funding Distribution (Platform vs Non-platform)")
axs[0].set_xlabel("Investment Type")
axs[0].set_ylabel("Total Funding (USD, log scale)")
axs[0].set_yscale("log")          # Funding is usually very skewed

# ---------------------------------------------------
# 2) KDE PLOT — Total Funding
# ---------------------------------------------------
sns.kdeplot(
    data=df_ipo,
    x='total_funding_usd',
    hue=platform_col,
    fill=True,
    common_norm=False,
    ax=axs[1]
)
axs[1].set_title("Distribution of Total Funding")
axs[1].set_xlabel("Total Funding (USD, log scale)")
axs[1].set_ylabel("Density")
axs[1].set_xscale("log")          # KDE also needs log-scale

# ---------------------------------------------------
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# ---------------------------------------------------
# 1) BOX PLOT — Years Since First Round
# ---------------------------------------------------
sns.boxplot(
    data=df_ipo,
    y='years_since_first_round',
    x=platform_col,
    ax=axs[0]
)
axs[0].set_title("Years Since First Funding Round (Platform vs Non-platform)")
axs[0].set_xlabel("Investment Type")
axs[0].set_ylabel("Years Since First Round")

# ---------------------------------------------------
# 2) KDE PLOT — Years Since First Round
# ---------------------------------------------------
sns.kdeplot(
    data=df_ipo,
    x='years_since_first_round',
    hue=platform_col,
    fill=True,
    common_norm=False,
    ax=axs[1]
)
axs[1].set_title("Distribution of Years Since First Funding Round")
axs[1].set_xlabel("Years Since First Round")
axs[1].set_ylabel("Density")

plt.tight_layout()
plt.show()

Conditional on being acquired, what is the 
- Amount of funding,
- Number of funding rounds,
- Years since first round for platform vs. non-platform businesses?

In [ ]:
df_acquired = df_org_grouped[['uuid', 'name','short_description', 'created_at', 'status','company_age','num_funding_rounds', 'first_round_on','total_funding_usd', 'num_exits', 'is_inactive', 'is_vc_funded', 'is_platform', 'is_digital_nonplatform']].copy()
df_acquired = df_acquired[df_acquired['status']=='acquired']
df_acquired['years_since_first_round'] = ((today - df_acquired['first_round_on']).dt.days / 365)
platform_col = 'is_platform'

In [ ]:
df_acquired[platform_col].value_counts()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 6))

# --- Boxplot ---
sns.boxplot(
    data=df_acquired,
    y='num_funding_rounds',
    x=platform_col,
    ax=axs[0]
)
axs[0].set_title("Funding rounds Distribution by Investment Type (Platform vs Non-platform) for acquired companies")
axs[0].set_xlabel("Investment Type")
axs[0].set_ylabel("number of rounds")

# --- KDE plot ---
sns.kdeplot(
    data=df_acquired,
    x='num_funding_rounds',
    hue=platform_col,
    fill=True,
    common_norm=False,
    ax=axs[1]
)
axs[1].set_title("Distribution of number of Funding Round of acquired company")
axs[1].set_xlabel("number of funding rounds")
axs[1].set_ylabel("Density")

plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# ---------------------------------------------------
# 1) BOX PLOT — Total Funding
# ---------------------------------------------------
sns.boxplot(
    data=df_acquired,
    y='total_funding_usd',
    x=platform_col,
    ax=axs[0]
)
axs[0].set_title("Total Funding Distribution (Platform vs Non-platform) for acquired companies")
axs[0].set_xlabel("Investment Type")
axs[0].set_ylabel("Total Funding (USD, log scale)")
axs[0].set_yscale("log")          # Funding is usually very skewed

# ---------------------------------------------------
# 2) KDE PLOT — Total Funding
# ---------------------------------------------------
sns.kdeplot(
    data=df_acquired,
    x='total_funding_usd',
    hue=platform_col,
    fill=True,
    common_norm=False,
    ax=axs[1]
)
axs[1].set_title("Distribution of Total Funding for acquired companies")
axs[1].set_xlabel("Total Funding (USD, log scale)")
axs[1].set_ylabel("Density")
axs[1].set_xscale("log")          # KDE also needs log-scale

# ---------------------------------------------------
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# ---------------------------------------------------
# 1) BOX PLOT — Years Since First Round
# ---------------------------------------------------
sns.boxplot(
    data=df_acquired,
    y='years_since_first_round',
    x=platform_col,
    ax=axs[0]
)
axs[0].set_title("Years Since First Funding Round (Platform vs Non-platform) for acquired companies")
axs[0].set_xlabel("Investment Type")
axs[0].set_ylabel("Years Since First Round")

# ---------------------------------------------------
# 2) KDE PLOT — Years Since First Round
# ---------------------------------------------------
sns.kdeplot(
    data=df_acquired,
    x='years_since_first_round',
    hue=platform_col,
    fill=True,
    common_norm=False,
    ax=axs[1]
)
axs[1].set_title("Distribution of Years Since First Funding Round for acquired companies")
axs[1].set_xlabel("Years Since First Round")
axs[1].set_ylabel("Density")

plt.tight_layout()
plt.show()